# Combine Extracted Kline CSVs By Time Range With Integrity Checks

This notebook uses **already extracted CSVs** and provides:
1. Time-range combine by month (`START_YYYY_MM` to `END_YYYY_MM`).
2. Column keep/drop control.
3. Output format selection (`csv` or `parquet`).
4. Data integrity checks for timestamp and OHLC consistency.
5. A summary integrity log in `.txt`.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv
import itertools
import re

# -----------------------------
# Configuration
# -----------------------------
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "downloads").exists() and (PROJECT_ROOT.parent / "downloads").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SYMBOL = "BTCUSDT"
INTERVAL = "1s"
START_YYYY_MM = "2018-01"  # inclusive
END_YYYY_MM = "2020-02"    # inclusive

EXTRACTED_DIR = PROJECT_ROOT / "downloads" / "test1"
OUTPUT_DIR = EXTRACTED_DIR

# Output options: 'csv' or 'parquet'
OUTPUT_FORMAT = "csv"
OUTPUT_NAME = ""  # empty means auto name
SUMMARY_LOG_NAME = ""  # empty means auto name

# Column selection options (choose one style):
# 1) KEEP_COLUMNS: keep only these columns
# 2) DROP_COLUMNS: drop these columns
# If KEEP_COLUMNS is not empty, DROP_COLUMNS is ignored and must stay empty.
KEEP_COLUMNS = []
DROP_COLUMNS = []

# Parquet options
PARQUET_COMPRESSION = "snappy"
PARQUET_CHUNK_SIZE = 200000

# Integrity log options
MAX_EXAMPLES_PER_ISSUE = 15

KLINE_COLUMNS = ['Open time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Close time', 'Quote asset volume', 'Number of trades', 'Taker buy base asset volume', 'Taker buy quote asset volume', 'Ignore']

print(f"Project root : {PROJECT_ROOT}")
print(f"Extracted dir: {EXTRACTED_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Range        : {START_YYYY_MM} to {END_YYYY_MM}")
print(f"Output format: {OUTPUT_FORMAT}")


In [ ]:
def parse_yyyy_mm(text: str) -> datetime:
    return datetime.strptime(text, "%Y-%m")


def interval_to_us(interval: str):
    # Returns expected bar size in microseconds; None for variable-size intervals like 1mo.
    mapping_seconds = {
        "1s": 1,
        "1m": 60,
        "3m": 3 * 60,
        "5m": 5 * 60,
        "15m": 15 * 60,
        "30m": 30 * 60,
        "1h": 60 * 60,
        "2h": 2 * 60 * 60,
        "4h": 4 * 60 * 60,
        "6h": 6 * 60 * 60,
        "8h": 8 * 60 * 60,
        "12h": 12 * 60 * 60,
        "1d": 24 * 60 * 60,
        "3d": 3 * 24 * 60 * 60,
        "1w": 7 * 24 * 60 * 60,
    }
    if interval == "1mo":
        return None
    sec = mapping_seconds.get(interval)
    if sec is None:
        raise ValueError(f"Unsupported interval for integrity checks: {interval}")
    return sec * 1_000_000


def parse_timestamp_to_us(raw_value: str):
    # Spot data may be milliseconds (historical) or microseconds (newer data).
    ts = int(raw_value)
    if ts >= 10**15:
        return ts, "us"
    if ts >= 10**12:
        return ts * 1000, "ms"
    if ts >= 10**9:
        return ts * 1_000_000, "s"
    return None, "unknown"


def resolve_output_columns(all_columns, keep_columns, drop_columns):
    keep = [c for c in keep_columns]
    drop = [c for c in drop_columns]

    unknown_keep = [c for c in keep if c not in all_columns]
    unknown_drop = [c for c in drop if c not in all_columns]
    if unknown_keep:
        raise ValueError(f"Unknown KEEP_COLUMNS entries: {unknown_keep}")
    if unknown_drop:
        raise ValueError(f"Unknown DROP_COLUMNS entries: {unknown_drop}")

    if keep and drop:
        raise ValueError("Use either KEEP_COLUMNS or DROP_COLUMNS, not both.")

    if keep:
        result = keep
    else:
        result = [c for c in all_columns if c not in set(drop)]

    if not result:
        raise ValueError("No columns selected.")

    if len(result) != len(set(result)):
        raise ValueError("Selected columns contain duplicates.")

    return result


def coerce_for_parquet(column_name: str, value: str):
    int_cols = {"Open time", "Close time", "Number of trades", "Ignore"}
    float_cols = {
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Quote asset volume",
        "Taker buy base asset volume",
        "Taker buy quote asset volume",
    }

    if value == "" or value is None:
        return None

    if column_name in int_cols:
        try:
            return int(value)
        except ValueError:
            return None

    if column_name in float_cols:
        try:
            return float(value)
        except ValueError:
            return None

    return value


start_dt = parse_yyyy_mm(START_YYYY_MM)
end_dt = parse_yyyy_mm(END_YYYY_MM)
if start_dt > end_dt:
    raise ValueError(f"START_YYYY_MM must be <= END_YYYY_MM ({START_YYYY_MM} > {END_YYYY_MM})")

if not EXTRACTED_DIR.exists():
    raise FileNotFoundError(f"Extracted directory not found: {EXTRACTED_DIR}")

output_format = OUTPUT_FORMAT.strip().lower()
if output_format not in {"csv", "parquet"}:
    raise ValueError("OUTPUT_FORMAT must be 'csv' or 'parquet'.")

selected_columns = resolve_output_columns(KLINE_COLUMNS, KEEP_COLUMNS, DROP_COLUMNS)
print("Selected columns:")
for c in selected_columns:
    print(f"  - {c}")

csv_pattern = re.compile(
    rf"^{re.escape(SYMBOL)}-{re.escape(INTERVAL)}-(\d{{4}}-\d{{2}})(?:-part(\d+))?\.csv$",
    flags=re.IGNORECASE,
)

selected_csvs = []
for csv_path in EXTRACTED_DIR.iterdir():
    if not csv_path.is_file() or csv_path.suffix.lower() != ".csv":
        continue
    m = csv_pattern.match(csv_path.name)
    if not m:
        continue

    month_text = m.group(1)
    part_text = m.group(2)
    part_num = int(part_text) if part_text else 1
    month_dt = parse_yyyy_mm(month_text)

    if start_dt <= month_dt <= end_dt:
        selected_csvs.append((month_dt, part_num, month_text, csv_path))

selected_csvs.sort(key=lambda x: (x[0], x[1], x[3].name))

if not selected_csvs:
    raise ValueError(
        f"No extracted CSV files matched range {START_YYYY_MM} to {END_YYYY_MM} in {EXTRACTED_DIR}"
    )

expected_interval_us = interval_to_us(INTERVAL)

print(f"Selected extracted CSV files: {len(selected_csvs)}")
for i, (_, part_num, month_text, csv_path) in enumerate(selected_csvs, start=1):
    if i <= 10 or i > len(selected_csvs) - 3:
        part_label = f" part{part_num}" if "-part" in csv_path.stem else ""
        print(f"  {month_text}{part_label}: {csv_path.name}")
    elif i == 11:
        print("  ...")


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

auto_output_name = f"{SYMBOL}-{INTERVAL}-{START_YYYY_MM}_to_{END_YYYY_MM}-combined.{output_format}"
output_name = OUTPUT_NAME if OUTPUT_NAME else auto_output_name
combined_output_path = OUTPUT_DIR / output_name

auto_log_name = f"{combined_output_path.stem}-integrity-summary.txt"
summary_log_name = SUMMARY_LOG_NAME if SUMMARY_LOG_NAME else auto_log_name
summary_log_path = OUTPUT_DIR / summary_log_name

issue_counts = Counter()
issue_examples = defaultdict(list)
timestamp_units = Counter()

rows_total = 0
rows_written = 0
files_processed = 0

def record_issue(issue_key: str, detail: str):
    issue_counts[issue_key] += 1
    if len(issue_examples[issue_key]) < MAX_EXAMPLES_PER_ISSUE:
        issue_examples[issue_key].append(detail)

csv_writer = None
csv_handle = None
parquet_writer = None
parquet_schema = None
parquet_buffer = None
buffer_size = 0

try:
    if output_format == "csv":
        csv_handle = combined_output_path.open("w", newline="", encoding="utf-8")
        csv_writer = csv.writer(csv_handle)
        csv_writer.writerow(selected_columns)
    else:
        try:
            import pyarrow as pa
            import pyarrow.parquet as pq
        except ImportError as exc:
            raise ImportError(
                "Parquet output requires pyarrow. Install it, then rerun."
            ) from exc

        parquet_field_types = []
        for col in selected_columns:
            if col in {"Open time", "Close time", "Number of trades", "Ignore"}:
                parquet_field_types.append((col, pa.int64()))
            elif col in {"Open", "High", "Low", "Close", "Volume", "Quote asset volume", "Taker buy base asset volume", "Taker buy quote asset volume"}:
                parquet_field_types.append((col, pa.float64()))
            else:
                parquet_field_types.append((col, pa.string()))

        parquet_schema = pa.schema(parquet_field_types)
        parquet_writer = pq.ParquetWriter(
            str(combined_output_path), parquet_schema, compression=PARQUET_COMPRESSION
        )
        parquet_buffer = {col: [] for col in selected_columns}

    prev_open_time_us = None

    for _, _, _, csv_path in selected_csvs:
        files_processed += 1

        with csv_path.open("r", newline="", encoding="utf-8") as src:
            reader = csv.reader(src)
            first_row = next(reader, None)
            if first_row is None:
                record_issue("empty_file", f"{csv_path.name}: file has no rows")
                continue

            if first_row == KLINE_COLUMNS:
                row_iter = reader
            else:
                record_issue("missing_or_unexpected_header", f"{csv_path.name}: first row is not expected header")
                row_iter = itertools.chain([first_row], reader)

            for file_row_idx, row in enumerate(row_iter, start=1):
                rows_total += 1

                if len(row) != len(KLINE_COLUMNS):
                    record_issue(
                        "malformed_row_column_count",
                        f"{csv_path.name}:{file_row_idx} has {len(row)} columns (expected {len(KLINE_COLUMNS)})",
                    )
                    continue

                row_map = dict(zip(KLINE_COLUMNS, row))

                # Timestamp checks
                open_us = None
                close_us = None

                try:
                    open_us, open_unit = parse_timestamp_to_us(row_map["Open time"])
                    timestamp_units[open_unit] += 1
                    if open_us is None:
                        record_issue(
                            "open_time_unknown_unit",
                            f"{csv_path.name}:{file_row_idx} open_time={row_map['Open time']}",
                        )
                except ValueError:
                    record_issue(
                        "open_time_parse_error",
                        f"{csv_path.name}:{file_row_idx} open_time={row_map['Open time']}",
                    )

                try:
                    close_us, close_unit = parse_timestamp_to_us(row_map["Close time"])
                    timestamp_units[close_unit] += 1
                    if close_us is None:
                        record_issue(
                            "close_time_unknown_unit",
                            f"{csv_path.name}:{file_row_idx} close_time={row_map['Close time']}",
                        )
                except ValueError:
                    record_issue(
                        "close_time_parse_error",
                        f"{csv_path.name}:{file_row_idx} close_time={row_map['Close time']}",
                    )

                if open_us is not None and prev_open_time_us is not None:
                    delta = open_us - prev_open_time_us
                    if delta < 0:
                        record_issue(
                            "timestamp_decrease",
                            f"{csv_path.name}:{file_row_idx} delta_us={delta}",
                        )
                    elif delta == 0:
                        record_issue(
                            "timestamp_duplicate",
                            f"{csv_path.name}:{file_row_idx} duplicate open time",
                        )
                    elif expected_interval_us is not None and delta != expected_interval_us:
                        record_issue(
                            "timestamp_step_mismatch",
                            f"{csv_path.name}:{file_row_idx} delta_us={delta}, expected={expected_interval_us}",
                        )
                        if delta > expected_interval_us:
                            record_issue(
                                "timestamp_gap",
                                f"{csv_path.name}:{file_row_idx} gap_us={delta - expected_interval_us}",
                            )
                        else:
                            record_issue(
                                "timestamp_overlap",
                                f"{csv_path.name}:{file_row_idx} overlap_us={expected_interval_us - delta}",
                            )

                if open_us is not None:
                    prev_open_time_us = open_us

                if open_us is not None and close_us is not None:
                    if close_us < open_us:
                        record_issue(
                            "close_before_open",
                            f"{csv_path.name}:{file_row_idx} open_us={open_us}, close_us={close_us}",
                        )
                    if expected_interval_us is not None:
                        expected_close = open_us + expected_interval_us - 1
                        if close_us != expected_close:
                            record_issue(
                                "close_time_mismatch",
                                f"{csv_path.name}:{file_row_idx} close_us={close_us}, expected_close={expected_close}",
                            )

                # OHLC checks
                try:
                    o = float(row_map["Open"])
                    h = float(row_map["High"])
                    l = float(row_map["Low"])
                    c = float(row_map["Close"])

                    if h < l:
                        record_issue(
                            "ohlc_high_below_low",
                            f"{csv_path.name}:{file_row_idx} high={h}, low={l}",
                        )
                    if h < max(o, c, l):
                        record_issue(
                            "ohlc_high_inconsistent",
                            f"{csv_path.name}:{file_row_idx} open={o}, high={h}, low={l}, close={c}",
                        )
                    if l > min(o, c, h):
                        record_issue(
                            "ohlc_low_inconsistent",
                            f"{csv_path.name}:{file_row_idx} open={o}, high={h}, low={l}, close={c}",
                        )
                except ValueError:
                    record_issue(
                        "ohlc_parse_error",
                        f"{csv_path.name}:{file_row_idx} open/high/low/close parse failed",
                    )

                # Additional sanity checks
                try:
                    if float(row_map["Volume"]) < 0:
                        record_issue("negative_volume", f"{csv_path.name}:{file_row_idx}")
                except ValueError:
                    record_issue("volume_parse_error", f"{csv_path.name}:{file_row_idx}")

                try:
                    if int(row_map["Number of trades"]) < 0:
                        record_issue("negative_trade_count", f"{csv_path.name}:{file_row_idx}")
                except ValueError:
                    record_issue("trades_parse_error", f"{csv_path.name}:{file_row_idx}")

                # Write output
                if output_format == "csv":
                    csv_writer.writerow([row_map[col] for col in selected_columns])
                    rows_written += 1
                else:
                    for col in selected_columns:
                        parquet_buffer[col].append(coerce_for_parquet(col, row_map[col]))
                    buffer_size += 1
                    rows_written += 1

                    if buffer_size >= PARQUET_CHUNK_SIZE:
                        table = pa.Table.from_pydict(parquet_buffer, schema=parquet_schema)
                        parquet_writer.write_table(table)
                        parquet_buffer = {col: [] for col in selected_columns}
                        buffer_size = 0

    if output_format == "parquet" and buffer_size > 0:
        table = pa.Table.from_pydict(parquet_buffer, schema=parquet_schema)
        parquet_writer.write_table(table)

finally:
    if csv_handle is not None:
        csv_handle.close()
    if parquet_writer is not None:
        parquet_writer.close()

run_finished_at = datetime.now(timezone.utc)

summary_lines = []
summary_lines.append("=== Kline Combine Integrity Summary ===")
summary_lines.append(f"Generated at (UTC): {run_finished_at.isoformat()}")
summary_lines.append("")
summary_lines.append("[Config]")
summary_lines.append(f"SYMBOL={SYMBOL}")
summary_lines.append(f"INTERVAL={INTERVAL}")
summary_lines.append(f"START_YYYY_MM={START_YYYY_MM}")
summary_lines.append(f"END_YYYY_MM={END_YYYY_MM}")
summary_lines.append(f"OUTPUT_FORMAT={output_format}")
summary_lines.append(f"EXTRACTED_DIR={EXTRACTED_DIR}")
summary_lines.append(f"OUTPUT_PATH={combined_output_path}")
summary_lines.append(f"SUMMARY_LOG_PATH={summary_log_path}")
summary_lines.append(f"EXPECTED_INTERVAL_US={expected_interval_us}")
summary_lines.append(f"SELECTED_COLUMNS={selected_columns}")
summary_lines.append("")
summary_lines.append("[Run Stats]")
summary_lines.append(f"FILES_SELECTED={len(selected_csvs)}")
summary_lines.append(f"FILES_PROCESSED={files_processed}")
summary_lines.append(f"ROWS_READ={rows_total}")
summary_lines.append(f"ROWS_WRITTEN={rows_written}")
summary_lines.append(f"TIMESTAMP_UNITS_OBSERVED={dict(timestamp_units)}")
summary_lines.append("")

summary_lines.append("[Issue Counts]")
if issue_counts:
    denom = rows_total if rows_total > 0 else 1
    for issue, count in issue_counts.most_common():
        pct = (count / denom) * 100
        summary_lines.append(f"{issue}: {count} ({pct:.6f}% of rows)")
else:
    summary_lines.append("No integrity issues detected by implemented checks.")

summary_lines.append("")
summary_lines.append("[Issue Examples]")
if issue_examples:
    for issue, examples in issue_examples.items():
        summary_lines.append(f"{issue}:")
        for ex in examples:
            summary_lines.append(f"  - {ex}")
else:
    summary_lines.append("No issue examples.")

summary_log_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

print(f"Combined output written: {combined_output_path}")
print(f"Summary log written    : {summary_log_path}")
print(f"Rows read              : {rows_total}")
print(f"Rows written           : {rows_written}")
print(f"Issues found           : {sum(issue_counts.values())}")


In [ ]:
print("Summary log preview (first 40 lines):")
with summary_log_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 39:
            break

if output_format == "csv":
    print("\nCombined CSV preview (first 6 lines):")
    with combined_output_path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            print(line.rstrip())
            if i >= 5:
                break
else:
    size_mb = combined_output_path.stat().st_size / (1024 * 1024)
    print(f"\nParquet file size: {size_mb:.2f} MB")
